### 사용 데이터
- 음식점: **상미규카츠, 파작, 몽탄, 그로어스, 아림당**
- 이동 수단: 자동차
- 가중치: 양방향 이동 시간의 평균(분)

| 출발 \ 도착 | 상미규카츠 | 파작 | 몽탄 | 그로어스 | 아림당 |
|:-----------:|:----------:|:----:|:----:|:--------:|:------:|
| 상미규카츠  | 0          | 23   | 42   | 39       | 43     |
| 파작        | 32         | 0    | 22   | 31       | 44     |
| 몽탄        | 35         | 20   | 0    | 37       | 25     |
| 그로어스    | 38         | 27   | 30   | 0        | 24     |
| 아림당      | 38         | 34   | 19   | 22       | 0      |

In [ ]:
# 라이브러리 임포트 및 환경 설정
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
import networkx as nx
import numpy as np
from collections import deque
import heapq

# 한글 폰트 설정
FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(FONT_PATH)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print('환경 설정 완료')

## 1. 그래프 데이터 및 기본 시각화

In [ ]:
# 음식점 노드 및 이동 시간 데이터 (단위: 분, 자동차 이동)
nodes = ['상미규카츠', '파작', '몽탄', '그로어스', '아림당']

# raw: {(출발, 도착): (정방향 시간, 역방향 시간)}
# 비방향 그래프이므로 양방향 평균을 가중치로 사용
raw = {
    ('상미규카츠','파작'): (23,32), ('상미규카츠','몽탄'): (42,35),
    ('상미규카츠','그로어스'): (39,38), ('상미규카츠','아림당'): (43,38),
    ('파작','몽탄'): (22,20), ('파작','그로어스'): (31,27),
    ('파작','아림당'): (44,34), ('몽탄','그로어스'): (37,30),
    ('몽탄','아림당'): (25,19), ('그로어스','아림당'): (24,22),
}

G = nx.Graph()
G.add_nodes_from(nodes)
for (u,v),(a,b) in raw.items():
    w = round((a+b)/2, 1)
    if not G.has_edge(u,v):
        G.add_edge(u,v, weight=w)

pos = nx.circular_layout(G)

# 색상 팔레트
C_BG='#1a1a2e'; C_NODE='#2d3561'; C_EDGE='#4a4e69'
C_VISITED='#e94560'; C_CURRENT='#f5a623'
C_ACTIVE_E='#00b4d8'; C_MST_E='#06d6a0'; C_TEXT='#ffffff'

print(f'노드 수: {G.number_of_nodes()}, 간선 수: {G.number_of_edges()}')
print('간선 목록(가중치):')
for u,v,d in sorted(G.edges(data=True), key=lambda x: x[2]['weight']):
    print(f'  {u} ↔ {v}: {d["weight"]}분')

In [ ]:
# 그래프 기본 시각화
fig, ax = plt.subplots(figsize=(9,7))
fig.patch.set_facecolor(C_BG)
ax.set_facecolor(C_BG)

nx.draw_networkx_edges(G, pos, ax=ax, edge_color=C_EDGE, width=2.0, alpha=0.8)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=C_NODE, node_size=2200,
                       edgecolors='#ffffff', linewidths=2)
nx.draw_networkx_labels(G, pos, ax=ax, font_color=C_TEXT, font_size=10,
                        font_weight='bold', font_family='NanumGothic')
edge_labels = nx.get_edge_attributes(G, 'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax,
                             font_color='#dddddd', font_size=9,
                             font_family='NanumGothic',
                             bbox=dict(boxstyle='round,pad=0.2',
                                       fc='#2d2d44', ec='none', alpha=0.8))
ax.set_title('음식점 그래프 (가중치: 자동차 이동 시간, 단위: 분)',
             color=C_TEXT, fontsize=13,
             fontproperties=fm.FontProperties(fname=FONT_PATH))
ax.axis('off')
plt.tight_layout()
plt.savefig('graph.png', dpi=150, facecolor=C_BG)
plt.show()
print('그래프 저장 완료: graph.png')

## 공통 유틸리티 함수

In [ ]:
def base_draw(ax, G, pos, node_colors=None, edge_colors=None, edge_widths=None, title=''):
    ax.set_facecolor(C_BG)
    if node_colors is None: node_colors = [C_NODE]*len(G.nodes())
    if edge_colors is None: edge_colors = [C_EDGE]*len(G.edges())
    if edge_widths is None: edge_widths = [1.5]*len(G.edges())
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors,
                           width=edge_widths, alpha=0.85)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                           node_size=2000, edgecolors='#ffffff', linewidths=1.5)
    nx.draw_networkx_labels(G, pos, ax=ax, font_color=C_TEXT,
                            font_size=9, font_weight='bold',
                            font_family='NanumGothic')
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax,
                                 font_color='#dddddd', font_size=8,
                                 font_family='NanumGothic',
                                 bbox=dict(boxstyle='round,pad=0.2',
                                           fc='#2d2d44', ec='none', alpha=0.8))
    ax.set_title(title, color=C_TEXT, fontsize=12, pad=10,
                 fontproperties=fm.FontProperties(fname=FONT_PATH))
    ax.axis('off')

def add_legend(ax, handles):
    ax.legend(handles=handles, loc='lower left',
              facecolor='#2d2d44', labelcolor=C_TEXT, fontsize=8,
              framealpha=0.85, prop=fm.FontProperties(fname=FONT_PATH))

print('공통 함수 정의 완료')

## 2. DFS (깊이 우선 탐색) 애니메이션

In [ ]:
def dfs_frames(start='상미규카츠'):
    frames = []
    visited, stack = [], [start]
    adj = {n: sorted(list(G.neighbors(n)), key=lambda x: G[n][x]['weight']) for n in nodes}
    frames.append({'visited':[], 'current':None, 'ae':None,
                   'title':'DFS 시작 (출발: 상미규카츠)'})
    while stack:
        node = stack.pop()
        if node in visited: continue
        visited.append(node)
        frames.append({'visited':list(visited), 'current':node, 'ae':None,
                       'title':f'DFS: {node} 방문 ({len(visited)}/{len(nodes)})'})
        for nbr in reversed(adj[node]):
            if nbr not in visited:
                frames.append({'visited':list(visited), 'current':node,
                               'ae':(node,nbr),
                               'title':f'DFS: {node} → {nbr} 탐색 중'})
                stack.append(nbr)
    frames.append({'visited':list(visited), 'current':None, 'ae':None,
                   'title':'DFS 완료! 모든 음식점 방문'})
    return frames

def make_dfs():
    frames = dfs_frames()
    fig, ax = plt.subplots(figsize=(8,6))
    fig.patch.set_facecolor(C_BG)
    def update(i):
        ax.clear()
        f = frames[i]
        nc = [C_CURRENT if n==f['current'] else C_VISITED if n in f['visited']
              else C_NODE for n in G.nodes()]
        ec, ew = [], []
        for u,v in G.edges():
            if f['ae'] and {u,v}=={f['ae'][0],f['ae'][1]}: ec.append(C_ACTIVE_E); ew.append(3.5)
            elif u in f['visited'] and v in f['visited']: ec.append(C_VISITED); ew.append(2.0)
            else: ec.append(C_EDGE); ew.append(1.5)
        base_draw(ax, G, pos, nc, ec, ew, f['title'])
        add_legend(ax, [
            mpatches.Patch(color=C_CURRENT, label='현재 노드'),
            mpatches.Patch(color=C_VISITED, label='방문 완료'),
            mpatches.Patch(color=C_NODE, label='미방문'),
            mpatches.Patch(color=C_ACTIVE_E, label='탐색 중인 간선'),
        ])
    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=900, repeat=False)
    ani.save('dfs.mp4', writer='ffmpeg', fps=1,
             extra_args=['-vcodec','libx264','-pix_fmt','yuv420p'])
    plt.close()
    print(f'dfs.mp4 저장 완료 (총 {len(frames)}프레임)')

make_dfs()

## 3. BFS (너비 우선 탐색) 애니메이션

In [ ]:
def bfs_frames(start='상미규카츠'):
    frames = []
    visited, queue = [start], deque([start])
    adj = {n: sorted(list(G.neighbors(n)), key=lambda x: G[n][x]['weight']) for n in nodes}
    frames.append({'visited':[start], 'current':start, 'queue':list(queue),
                   'ae':None, 'title':'BFS 시작 (출발: 상미규카츠)'})
    while queue:
        node = queue.popleft()
        frames.append({'visited':list(visited), 'current':node,
                       'queue':list(queue), 'ae':None,
                       'title':f'BFS: {node} 처리 중 ({len(visited)}/{len(nodes)})'})
        for nbr in adj[node]:
            if nbr not in visited:
                visited.append(nbr); queue.append(nbr)
                frames.append({'visited':list(visited), 'current':node,
                               'queue':list(queue), 'ae':(node,nbr),
                               'title':f'BFS: {node} → {nbr} 큐에 추가'})
    frames.append({'visited':list(visited), 'current':None, 'queue':[], 'ae':None,
                   'title':'BFS 완료! 모든 음식점 방문'})
    return frames

def make_bfs():
    frames = bfs_frames()
    fig, ax = plt.subplots(figsize=(8,6))
    fig.patch.set_facecolor(C_BG)
    def update(i):
        ax.clear()
        f = frames[i]
        q = f['queue']
        nc = [C_CURRENT if n==f['current'] else '#7b2d8b' if n in q
              else C_VISITED if n in f['visited'] else C_NODE for n in G.nodes()]
        ec, ew = [], []
        for u,v in G.edges():
            if f['ae'] and {u,v}=={f['ae'][0],f['ae'][1]}: ec.append(C_ACTIVE_E); ew.append(3.5)
            elif u in f['visited'] and v in f['visited']: ec.append(C_VISITED); ew.append(2.0)
            else: ec.append(C_EDGE); ew.append(1.5)
        base_draw(ax, G, pos, nc, ec, ew, f['title'])
        if q:
            fp = fm.FontProperties(fname=FONT_PATH)
            ax.text(0.5, -0.05, f'큐: {" → ".join(q)}', transform=ax.transAxes,
                    ha='center', color='#aaaaaa', fontsize=8, fontproperties=fp,
                    bbox=dict(fc='#2d2d44', ec='none', alpha=0.7))
        add_legend(ax, [
            mpatches.Patch(color=C_CURRENT, label='현재 노드'),
            mpatches.Patch(color='#7b2d8b', label='큐 대기 중'),
            mpatches.Patch(color=C_VISITED, label='방문 완료'),
            mpatches.Patch(color=C_NODE, label='미방문'),
            mpatches.Patch(color=C_ACTIVE_E, label='탐색 중인 간선'),
        ])
    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=900, repeat=False)
    ani.save('bfs.mp4', writer='ffmpeg', fps=1,
             extra_args=['-vcodec','libx264','-pix_fmt','yuv420p'])
    plt.close()
    print(f'bfs.mp4 저장 완료 (총 {len(frames)}프레임)')

make_bfs()

## 4. Prim 알고리즘 (MST) 애니메이션

In [ ]:
def prim_frames(start='상미규카츠'):
    frames = []
    in_mst, mst_edges, total_w = [start], [], 0
    frames.append({'in_mst':[start], 'mst_edges':[], 'cand':None, 'total':0,
                   'title':f'Prim 시작: {start} 선택'})
    while len(in_mst) < len(nodes):
        heap = []
        for u in in_mst:
            for v in G.neighbors(u):
                if v not in in_mst:
                    heapq.heappush(heap, (G[u][v]['weight'], u, v))
        while heap:
            w, u, v = heapq.heappop(heap)
            if v not in in_mst:
                frames.append({'in_mst':list(in_mst), 'mst_edges':list(mst_edges),
                               'cand':(u,v,w), 'total':total_w,
                               'title':f'Prim: 최소 간선 {u}↔{v} ({w}분) 선택'})
                in_mst.append(v); mst_edges.append((u,v,w)); total_w += w
                frames.append({'in_mst':list(in_mst), 'mst_edges':list(mst_edges),
                               'cand':None, 'total':total_w,
                               'title':f'Prim: {v} MST 추가 (누적: {total_w}분)'})
                break
    frames.append({'in_mst':list(in_mst), 'mst_edges':list(mst_edges),
                   'cand':None, 'total':total_w,
                   'title':f'Prim 완료! MST 총 이동시간: {total_w}분'})
    return frames

def make_prim():
    frames = prim_frames()
    fig, ax = plt.subplots(figsize=(8,6))
    fig.patch.set_facecolor(C_BG)
    def update(i):
        ax.clear()
        f = frames[i]
        nc = [C_VISITED if n in f['in_mst'] else C_NODE for n in G.nodes()]
        mst_set = {(u,v) for u,v,_ in f['mst_edges']}
        ec, ew = [], []
        for u,v in G.edges():
            if f['cand'] and {u,v}=={f['cand'][0],f['cand'][1]}: ec.append(C_CURRENT); ew.append(4.0)
            elif (u,v) in mst_set or (v,u) in mst_set: ec.append(C_MST_E); ew.append(3.0)
            else: ec.append(C_EDGE); ew.append(1.5)
        base_draw(ax, G, pos, nc, ec, ew, f['title'])
        fp = fm.FontProperties(fname=FONT_PATH)
        ax.text(0.5, -0.05, f'MST 총 이동시간: {f["total"]}분', transform=ax.transAxes,
                ha='center', color='#aaaaaa', fontsize=9, fontproperties=fp,
                bbox=dict(fc='#2d2d44', ec='none', alpha=0.7))
        add_legend(ax, [
            mpatches.Patch(color=C_VISITED, label='MST 포함 노드'),
            mpatches.Patch(color=C_NODE, label='미포함 노드'),
            mpatches.Patch(color=C_MST_E, label='MST 간선'),
            mpatches.Patch(color=C_CURRENT, label='선택 중인 간선'),
        ])
    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000, repeat=False)
    ani.save('prim.mp4', writer='ffmpeg', fps=1,
             extra_args=['-vcodec','libx264','-pix_fmt','yuv420p'])
    plt.close()
    print(f'prim.mp4 저장 완료 (총 {len(frames)}프레임)')

make_prim()

## 5. Kruskal 알고리즘 (MST) 애니메이션

In [ ]:
def kruskal_frames():
    frames = []
    parent = {n: n for n in nodes}
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(x, y):
        px, py = find(x), find(y)
        if px == py: return False
        parent[px] = py; return True

    sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'])
    mst_edges, skipped, total_w = [], [], 0
    frames.append({'mst_edges':[], 'skipped':[], 'cand':None, 'total':0,
                   'title':'Kruskal 시작: 간선을 가중치 순으로 정렬'})
    for u, v, d in sorted_edges:
        w = d['weight']
        frames.append({'mst_edges':list(mst_edges), 'skipped':list(skipped),
                       'cand':(u,v,w), 'total':total_w,
                       'title':f'Kruskal: 간선 {u}↔{v} ({w}분) 검토'})
        if union(u, v):
            mst_edges.append((u,v,w)); total_w += w
            frames.append({'mst_edges':list(mst_edges), 'skipped':list(skipped),
                           'cand':None, 'total':total_w,
                           'title':f'Kruskal: {u}↔{v} 추가! (누적: {total_w}분)'})
        else:
            skipped.append((u,v))
            frames.append({'mst_edges':list(mst_edges), 'skipped':list(skipped),
                           'cand':None, 'total':total_w,
                           'title':f'Kruskal: {u}↔{v} 사이클 형성 → 제외'})
        if len(mst_edges) == len(nodes)-1: break
    frames.append({'mst_edges':list(mst_edges), 'skipped':list(skipped),
                   'cand':None, 'total':total_w,
                   'title':f'Kruskal 완료! MST 총 이동시간: {total_w}분'})
    return frames

def make_kruskal():
    frames = kruskal_frames()
    fig, ax = plt.subplots(figsize=(8,6))
    fig.patch.set_facecolor(C_BG)
    def update(i):
        ax.clear()
        f = frames[i]
        mst_nodes = set()
        for u,v,_ in f['mst_edges']: mst_nodes.add(u); mst_nodes.add(v)
        nc = [C_VISITED if n in mst_nodes else C_NODE for n in G.nodes()]
        mst_set = {(u,v) for u,v,_ in f['mst_edges']}
        skip_set = {(u,v) for u,v in f['skipped']}
        ec, ew = [], []
        for u,v in G.edges():
            if f['cand'] and {u,v}=={f['cand'][0],f['cand'][1]}: ec.append(C_CURRENT); ew.append(4.0)
            elif (u,v) in mst_set or (v,u) in mst_set: ec.append(C_MST_E); ew.append(3.0)
            elif (u,v) in skip_set or (v,u) in skip_set: ec.append('#555555'); ew.append(1.0)
            else: ec.append(C_EDGE); ew.append(1.5)
        base_draw(ax, G, pos, nc, ec, ew, f['title'])
        fp = fm.FontProperties(fname=FONT_PATH)
        ax.text(0.5, -0.05,
                f'MST 총 이동시간: {f["total"]}분  |  MST 간선: {len(f["mst_edges"])}/{len(nodes)-1}',
                transform=ax.transAxes, ha='center', color='#aaaaaa',
                fontsize=9, fontproperties=fp,
                bbox=dict(fc='#2d2d44', ec='none', alpha=0.7))
        add_legend(ax, [
            mpatches.Patch(color=C_VISITED, label='MST 포함 노드'),
            mpatches.Patch(color=C_MST_E, label='MST 간선'),
            mpatches.Patch(color=C_CURRENT, label='검토 중인 간선'),
            mpatches.Patch(color='#555555', label='제외 간선(사이클)'),
        ])
    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000, repeat=False)
    ani.save('kruskal.mp4', writer='ffmpeg', fps=1,
             extra_args=['-vcodec','libx264','-pix_fmt','yuv420p'])
    plt.close()
    print(f'kruskal.mp4 저장 완료 (총 {len(frames)}프레임)')

make_kruskal()

In [ ]:
print("=== 모든 파일 생성 완료 ===")
print("dfs.mp4, bfs.mp4, prim.mp4, kruskal.mp4")